# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zain2502/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

---
**⚠️ SETUP — edit this before running:** this notebook assumes your data is a CSV export with columns like
`date, query, page, clicks, impressions, ctr, position`. Update `DATA_PATH` and the column names in the
first code cell below to match your actual export. Every number in this notebook is computed live from
that file — nothing here is a placeholder claim, so if you change the source, re-run top to bottom.

In [ ]:
# --- SETUP ---
# No real export was provided, so this cell GENERATES a synthetic Search Console-style
# dataset and writes it to DATA_PATH, then reads it back in exactly the way you'd read
# a real export. Swap this generation block out for pd.read_csv("your_real_file.csv")
# once you have actual data -- everything below this cell doesn't care which one you use.
import pandas as pd
import numpy as np
import os

DATA_PATH = "data/search_console_export.csv"

COL_DATE = "date"
COL_QUERY = "query"
COL_PAGE = "page"
COL_CLICKS = "clicks"
COL_IMPRESSIONS = "impressions"
COL_CTR = "ctr"
COL_POSITION = "position"

rng = np.random.default_rng(seed=42)

N_QUERIES = 40
N_PAGES = 15
N_DAYS = 28

queries = [f"query_{i:02d}" for i in range(N_QUERIES)]
pages = [f"/page-{i:02d}" for i in range(N_PAGES)]
dates = pd.date_range("2026-06-01", periods=N_DAYS, freq="D")

# Not every query x page x date combo exists in real GSC data -- sample a realistic
# sparse subset rather than the full cross product.
n_rows = 3000
sim_date = rng.choice(dates, size=n_rows)
sim_query = rng.choice(queries, size=n_rows)
sim_page = rng.choice(pages, size=n_rows)

sim_position = np.round(rng.gamma(shape=2.0, scale=5.0, size=n_rows) + 1, 1)
sim_position = np.clip(sim_position, 1, 100)

base_impressions = rng.poisson(lam=200, size=n_rows)
# higher (worse) position -> fewer impressions surfaced to users, roughly
sim_impressions = np.maximum(1, (base_impressions * (30 / sim_position)).astype(int))

# CTR falls off with position, plus noise
expected_ctr = np.clip(0.35 / sim_position, 0.001, 0.6)
noisy_ctr = np.clip(expected_ctr + rng.normal(0, 0.01, size=n_rows), 0, 1)
sim_clicks = np.minimum(sim_impressions, rng.binomial(sim_impressions, noisy_ctr))
sim_ctr = np.where(sim_impressions > 0, sim_clicks / sim_impressions, 0.0)

raw = pd.DataFrame({
    COL_DATE: sim_date,
    COL_QUERY: sim_query,
    COL_PAGE: sim_page,
    COL_CLICKS: sim_clicks,
    COL_IMPRESSIONS: sim_impressions,
    COL_CTR: np.round(sim_ctr, 4),
    COL_POSITION: sim_position,
})

# Collapse any accidental duplicate (date, query, page) combos the way a real GSC
# export already would be (it reports one aggregated row per grain key).
raw = (raw.groupby([COL_DATE, COL_QUERY, COL_PAGE], as_index=False)
          .agg({COL_CLICKS: "sum", COL_IMPRESSIONS: "sum", COL_POSITION: "mean"}))
raw[COL_POSITION] = raw[COL_POSITION].round(1)
raw[COL_CTR] = np.round(np.where(raw[COL_IMPRESSIONS] > 0, raw[COL_CLICKS] / raw[COL_IMPRESSIONS], 0.0), 4)

# Sprinkle a few missing values in, since real exports are rarely perfectly clean
missing_idx = rng.choice(raw.index, size=max(1, len(raw)//200), replace=False)
raw.loc[missing_idx, COL_POSITION] = np.nan

os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
raw.to_csv(DATA_PATH, index=False)

# Read it back the same way you'd read a real export
df = pd.read_csv(DATA_PATH, parse_dates=[COL_DATE])
print(df.shape)
df.head()

(2754, 7)


,date,query,page,clicks,impressions,position,ctr
0,2026-06-01,query_01,/page-03,15,531,12.2,0.0282
1,2026-06-01,query_01,/page-12,325,2472,2.5,0.1315
2,2026-06-01,query_02,/page-00,0,266,20.3,0.0000
3,2026-06-01,query_02,/page-09,326,2237,2.4,0.1457
4,2026-06-01,query_03,/page-02,231,2207,2.8,0.1047


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Claim:** one row = one (query, page, date) combination — the finest grain Search Console exposes.
Clicks, impressions, and position are aggregated by Google at this grain before export; anything coarser
(e.g. query-only, page-only) is a rollup of this table, not a separate source.

**Time window claim:** the export covers a fixed date range with daily granularity (no gaps assumed —
verified below, not asserted).

In [ ]:
# Verify: does (date, query, page) uniquely identify a row?
grain_cols = [COL_DATE, COL_QUERY, COL_PAGE]
n_rows = len(df)
n_unique_grain = df.drop_duplicates(subset=grain_cols).shape[0]
print(f"Total rows: {n_rows}")
print(f"Unique (date, query, page) combos: {n_unique_grain}")
print(f"Duplicate grain rows: {n_rows - n_unique_grain}")

# Verify: date window
print(f"Date range: {df[COL_DATE].min()} to {df[COL_DATE].max()}")
print(f"Distinct dates present: {df[COL_DATE].nunique()}")
expected_days = (df[COL_DATE].max() - df[COL_DATE].min()).days + 1
print(f"Expected days in range: {expected_days} | Actually observed: {df[COL_DATE].nunique()} "
      f"| Missing days: {expected_days - df[COL_DATE].nunique()}")

Total rows: 2754
Unique (date, query, page) combos: 2754
Duplicate grain rows: 0
Date range: 2026-06-01 00:00:00 to 2026-06-28 00:00:00
Distinct dates present: 28
Expected days in range: 28 | Actually observed: 28 | Missing days: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `impressions` | feature | observed demand signal, available pre-click |
| `position` | feature | observed ranking signal, available pre-click |
| `query`, `page` | context | identifies the row, not predictive on its own without encoding |
| `date` | context | needed for windowing/trend checks, not a per-row predictor as-is |
| `clicks` | label | the outcome being explained/predicted |
| `ctr` | excluded | it's a derived ratio of clicks/impressions — including it alongside both leaks the label |

Edit this table to match your actual columns once you've loaded the real export.

In [ ]:
# Verify the bucket assignment mechanically: flag any column that is a deterministic
# function of the label (clicks) combined with another feature -- classic leakage check.
if COL_CTR in df.columns:
    implied_ctr = (df[COL_CLICKS] / df[COL_IMPRESSIONS].replace(0, np.nan))
    diff = (df[COL_CTR] - implied_ctr).abs()
    print(f"Max |reported ctr - clicks/impressions|: {diff.max():.6f}")
    print("-> confirms ctr is derived from the label; excluding it as a feature is correct.")

feature_cols = [COL_IMPRESSIONS, COL_POSITION]
context_cols = [COL_QUERY, COL_PAGE, COL_DATE]
label_col = COL_CLICKS
excluded_cols = [COL_CTR]
print("feature:", feature_cols)
print("context:", context_cols)
print("label:", label_col)
print("excluded:", excluded_cols)

Max |reported ctr - clicks/impressions|: 0.000050
-> confirms ctr is derived from the label; excluding it as a feature is correct.
feature: ['impressions', 'position']
context: ['query', 'page', 'date']
label: clicks
excluded: ['ctr']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Row counts and missing values per column
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique()
})
summary

,dtype,n_missing,pct_missing,n_unique
date,datetime64[us],0,0.00,28
query,str,0,0.00,40
page,str,0,0.00,15
clicks,int64,0,0.00,372
impressions,int64,0,0.00,1343
position,float64,13,0.47,311
ctr,float64,0,0.00,1061


In [ ]:
# Sanity ranges on numeric fields Search Console guarantees are bounded
checks = {}
if COL_CTR in df.columns:
    checks["ctr_out_of_[0,1]"] = int(((df[COL_CTR] < 0) | (df[COL_CTR] > 1)).sum())
checks["clicks_negative"] = int((df[COL_CLICKS] < 0).sum())
checks["impressions_negative"] = int((df[COL_IMPRESSIONS] < 0).sum())
checks["clicks_gt_impressions"] = int((df[COL_CLICKS] > df[COL_IMPRESSIONS]).sum())
checks["position_lt_1"] = int((df[COL_POSITION] < 1).sum())
pd.Series(checks, name="count_of_violations")

ctr_out_of_[0,1]         0
clicks_negative          0
impressions_negative     0
clicks_gt_impressions    0
position_lt_1            0
Name: count_of_violations, dtype: int64

In [ ]:
# Per-day row counts -- spot sparse or overloaded days, useful for spotting collection gaps
daily_counts = df.groupby(COL_DATE).size().rename("n_rows")
daily_counts.describe()

count     28.000000
mean      98.357143
std       10.552213
min       77.000000
25%       89.000000
50%       98.500000
75%      104.250000
max      117.000000
Name: n_rows, dtype: float64

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **No causality**: impressions and position are correlational with clicks, not causal — a rank change
  and a click change happening together doesn't prove one caused the other.
- **Position is an average, not a distribution**: Search Console reports mean position per row: two
  rows with identical average position can have very different volatility across impressions.
- **No query intent or SERP-feature context**: this export doesn't say whether a query triggered a
  featured snippet, image pack, or ad — all of which change expected CTR at a given position.
- **Truncation for low-volume queries**: Search Console omits or anonymizes very-low-impression queries,
  so the tail of rare queries is systematically undercounted, not missing at random.
- **Window edges are soft**: the last 1-3 days of any live export are typically still being finalized by
  Google and may under-report until the data settles.

Claims here should stay in the vocabulary of *observed, measured, directional, decision-support* —
not causal or predictive certainty.

In [ ]:
# Verify the truncation/undercount claim: how much of total impression volume comes from
# the long tail of low-impression query-page rows?
sorted_impr = df[COL_IMPRESSIONS].sort_values(ascending=False).reset_index(drop=True)
total = sorted_impr.sum()
tail_share = sorted_impr[sorted_impr <= sorted_impr.quantile(0.5)].sum() / total if total else float("nan")
print(f"Share of total impressions held by the bottom 50% of rows (by impressions): {tail_share:.2%}")

# Verify the 'recent days may be incomplete' claim by comparing row counts of the last 3 days
# vs the median day
last_days = daily_counts.sort_index().tail(3)
print("Row counts for the most recent 3 days in the window:")
print(last_days)
print(f"Median daily row count across full window: {daily_counts.median()}")

Share of total impressions held by the bottom 50% of rows (by impressions): 23.71%
Row counts for the most recent 3 days in the window:
date
2026-06-26     95
2026-06-27    114
2026-06-28     98
Name: n_rows, dtype: int64
Median daily row count across full window: 98.5


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.